# Projekt supervised ML sample — development 2011–2022

**Cel.** Rozstrzygnięcie polityki supervised sample przed implementacją
preprocessingu i cross-validation. Notebook nie trenuje modeli, nie
imputuje, nie winsoryzuje, nie skaluje i nie wybiera cech.

**Niezmienne wejścia.** Frozen target `target_candidate_v2_pit_b v1.0.0`,
frozen historical research universe `v1.1.0` oraz frozen raw
point-in-time `X_t v1.0.0` są wejściami tylko do odczytu. Target jest
dołączany dopiero po konstrukcji `X_t`; jego status, klasa i provenance
nigdy nie są predictorami.

**Granica temporalna.** Analiza wartości i statusów obejmuje wyłącznie
feature years 2011–2022: train 2011–2020 i validation 2021–2022.
Wiersze 2023–2024 nie są zachowywane w ramce analitycznej ani używane w
decyzjach. Pełne pliki są skanowane bajtowo jedynie w celu sprawdzenia
zamrożonych hashy.

**Reprodukowalność.** Wszystkie transformacje diagnostyczne są czystymi
funkcjami. Jedna jawna komórka buduje niemutowaną dalej ramkę
`analysis_frame`; kolejne komórki wyłącznie tworzą nowe zestawienia.
Notebook powinien przechodzić `Restart & Run All` bez ręcznej zmiany
kolejności komórek.


In [1]:
from pathlib import Path
import csv
import hashlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 72)

DEV_YEAR_MIN, DEV_YEAR_MAX = 2011, 2022
STATUS_ORDER = [
    "available_core",
    "partially_available",
    "missing",
    "ambiguous",
    "not_available_non_xbrl",
]
FEATURE_BLOCKS = {
    "L": [
        "log_assets_t", "roa_t", "ocf_to_assets_t", "current_ratio_t",
        "liabilities_to_assets_t", "working_capital_to_assets_t",
        "accruals_to_assets_t",
    ],
    "D": [
        "asset_growth_1y", "delta_roa_1y", "delta_ocf_to_assets_1y",
        "current_ratio_change_1y", "delta_liabilities_to_assets_1y",
    ],
    "R": [
        "log1p_revenues_t", "profit_margin_t", "ocf_margin_t",
        "asset_turnover_t", "revenue_growth_1y",
    ],
}
FEATURES = [name for block in FEATURE_BLOCKS.values() for name in block]

def project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs/x_t_pit_v1_freeze_manifest.yaml").is_file():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu głównego projektu.")

ROOT = project_root()
PATHS = {
    "universe": ROOT / "data/processed/research_universe_pit.csv",
    "target": ROOT / "data/interim/target_candidate_v2_pit_b.csv",
    "x_t": ROOT / "data/processed/x_t_pit_v1_raw.csv",
    "target_application": ROOT / "data/processed/research_universe_pit_v1_1_0_target_pit_b_v1_0_0.csv",
}
EXPECTED = {
    "universe": {
        "sha256": "a449c8145d1f46f954f12b1dfc079bb0b367c4f7f5edf3332a983ad7c1fb8182",
        "rows": 103099, "columns": 81,
    },
    "target": {
        "sha256": "473aa403dfd15822a15ce985f7698efe4a4e3a66bcf30b7634f0ca646805e0ff",
        "rows": 26917, "columns": 802,
    },
    "x_t": {
        "sha256": "0f1b35b9ffbb1fb1c1cdfb7dff12e3efd8fb38f60b33407ff2b2a8fb6b88397f",
        "rows": 64901, "columns": 1072,
    },
    "target_application": {
        "sha256": "ea42eb43018b2c8e238e2c4757260bb692e27edd5429628e28892f360f0f7f7d",
        "rows": 64901, "columns": 832,
    },
}

def fingerprint_csv(path: Path) -> dict:
    digest = hashlib.sha256()
    byte_count = 0
    newline_count = 0
    with path.open("rb") as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
            byte_count += len(chunk)
            newline_count += chunk.count(b"\n")
    with path.open(newline="", encoding="utf-8") as handle:
        columns = len(next(csv.reader(handle)))
    return {
        "sha256": digest.hexdigest(),
        "rows": max(newline_count - 1, 0),
        "columns": columns,
        "bytes": byte_count,
    }

def read_development(path: Path, usecols: list[str]) -> pd.DataFrame:
    # Każdy chunk jest natychmiast ograniczany do development. Test rows
    # nie są zachowywane ani agregowane.
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=10_000, low_memory=False):
        years = pd.to_numeric(chunk["feature_year"], errors="coerce")
        parts.append(chunk.loc[years.between(DEV_YEAR_MIN, DEV_YEAR_MAX)].copy())
    return pd.concat(parts, ignore_index=True)

def add_size_group(frame: pd.DataFrame) -> pd.Series:
    result = pd.Series("missing_or_nonpositive", index=frame.index, dtype="object")
    observed = frame["current_t_assets_value"].gt(0) & frame["current_t_assets_value"].notna()
    for _, indexes in frame.loc[observed].groupby("feature_year", sort=True).groups.items():
        percentile = frame.loc[indexes, "current_t_assets_value"].rank(method="average", pct=True)
        result.loc[indexes] = pd.cut(
            percentile,
            bins=[0.0, 0.25, 0.50, 0.75, 1.0],
            labels=["Q1", "Q2", "Q3", "Q4"],
            include_lowest=True,
        ).astype(str)
    return result

def build_analysis_frame() -> pd.DataFrame:
    x_columns = [
        "research_universe_company_year_id", "cik10", "feature_year", "split",
        "research_sector", "historical_sic", "membership_status",
        "registrant_role_resolved", "economic_group_id", "x_t_status",
        "x_t_status_reason", "statement_scope_xbrl_reason",
        "anchor_period_validation_reason", "current_t_assets_value",
        "current_t_assets_status", "current_t_assets_reason",
        "L_available_count", "D_available_count", "R_available_count",
        "feature_available_count", "feature_missing_count",
        "feature_ambiguous_count", "feature_not_computable_count",
    ]
    for feature in FEATURES:
        x_columns.extend([f"{feature}_value", f"{feature}_status", f"{feature}_reason"])

    target_columns = [
        "research_universe_company_year_id", "feature_year", "split",
        "membership_status", "target_status", "target_candidate_v2_pit_b",
        "target_available", "target_application_reason",
    ]
    x_frame = read_development(PATHS["x_t"], x_columns)
    target_frame = read_development(PATHS["target_application"], target_columns).rename(
        columns={
            "feature_year": "target_feature_year",
            "split": "target_split",
            "membership_status": "target_membership_status",
        }
    )
    merged = x_frame.merge(
        target_frame,
        on="research_universe_company_year_id",
        how="left",
        validate="one_to_one",
    )
    assert len(merged) == 56_903
    assert merged["research_universe_company_year_id"].is_unique
    assert merged["feature_year"].between(DEV_YEAR_MIN, DEV_YEAR_MAX).all()
    assert merged["membership_status"].eq("eligible").all()
    assert merged["target_membership_status"].eq("eligible").all()
    assert merged["feature_year"].eq(merged["target_feature_year"]).all()
    assert merged["split"].eq(merged["target_split"]).all()
    assert merged["target_status"].notna().all()
    assert set(merged["split"].unique()) == {"train", "validation"}
    assert set(merged["x_t_status"].unique()) == set(STATUS_ORDER)
    assert merged.loc[
        ~merged["target_status"].eq("available"), "target_candidate_v2_pit_b"
    ].isna().all()

    x_profile = np.select(
        [
            merged["x_t_status"].eq("available_core"),
            merged["x_t_status"].eq("partially_available"),
        ],
        ["available_core", "partially_available"],
        default="no_observed_feature",
    )
    result = merged.assign(
        supervised_target_available=merged["target_status"].eq("available"),
        x_has_any_observed_feature=merged["x_t_status"].isin(
            ["available_core", "partially_available"]
        ),
        x_core_complete=merged["x_t_status"].eq("available_core"),
        x_observation_profile=x_profile,
    )
    return result.assign(size_group=add_size_group(result))

def show(title: str, frame: pd.DataFrame | pd.Series) -> None:
    print(f"\n{title}")
    print("=" * len(title))
    print(frame.to_string())

fingerprints = {name: fingerprint_csv(path) for name, path in PATHS.items()}
for name, actual in fingerprints.items():
    expected = EXPECTED[name]
    assert actual["sha256"] == expected["sha256"], f"Hash mismatch: {name}"
    assert actual["rows"] == expected["rows"], f"Row mismatch: {name}"
    assert actual["columns"] == expected["columns"], f"Column mismatch: {name}"

analysis_frame = build_analysis_frame()
supervised_frame = analysis_frame.loc[
    analysis_frame["supervised_target_available"]
].copy()

integrity = pd.DataFrame.from_dict(fingerprints, orient="index")
integrity["sha256_ok"] = [
    fingerprints[name]["sha256"] == EXPECTED[name]["sha256"] for name in integrity.index
]
integrity["shape_ok"] = [
    fingerprints[name]["rows"] == EXPECTED[name]["rows"]
    and fingerprints[name]["columns"] == EXPECTED[name]["columns"]
    for name in integrity.index
]
integrity["MiB"] = (integrity["bytes"] / 1024**2).round(1)
show("Kontrola integralności wejść", integrity[["rows", "columns", "MiB", "sha256_ok", "shape_ok"]])
print(f"\nDevelopment frame: {len(analysis_frame):,} rows; supervised ledger: {len(supervised_frame):,} rows")



Kontrola integralności wejść
                      rows  columns    MiB  sha256_ok  shape_ok
universe            103099       81  106.4       True      True
target               26917      802  169.3       True      True
x_t                  64901     1072  776.7       True      True
target_application   64901      832  383.7       True      True

Development frame: 56,903 rows; supervised ledger: 23,332 rows


## 1. Struktura dostępnych danych

`research_universe_company_year_id` jest kanonicznym kluczem 1:1.
Frozen raw `X_t` zawiera wyłącznie eligible company-years, a audited
target-application artifact mechanicznie stosuje frozen target do
frozen universe. Z target-application pobierane są wyłącznie status i
etykieta; pola D1–D5, target provenance i przyczyny targetu nie wchodzą
do macierzy cech.


In [2]:
structure = pd.DataFrame(
    [
        {
            "artefakt": "historical universe v1.1.0",
            "jednostka": "registrant–fiscal year; wszystkie statusy membership",
            "pełny_kształt": "103099 × 81",
            "rola_w_analizie": "upstream invariant; eligible membership",
        },
        {
            "artefakt": "target PIT-B v1.0.0",
            "jednostka": "company-year starego development frame",
            "pełny_kształt": "26917 × 802",
            "rola_w_analizie": "upstream invariant definicji targetu",
        },
        {
            "artefakt": "raw X_t v1.0.0",
            "jednostka": "eligible company-year",
            "pełny_kształt": "64901 × 1072",
            "rola_w_analizie": "metadata, 17 raw features, statusy, reason codes, provenance",
        },
        {
            "artefakt": "audited universe × target application",
            "jednostka": "eligible company-year",
            "pełny_kształt": "64901 × 832",
            "rola_w_analizie": "target_status i target_candidate_v2_pit_b po joinie 1:1",
        },
    ]
).set_index("artefakt")
schema = pd.DataFrame(
    [
        ("klucz i split", "research_universe_company_year_id, feature_year, split"),
        ("membership", "membership_status == eligible (assertion)"),
        ("metadata", "CIK, historical SIC/sector, registrant role, economic_group_id"),
        ("X row status", "x_t_status, x_t_status_reason"),
        ("X block counts", "L/D/R_available_count oraz łączny feature_available_count"),
        ("17 frozen features", "value + status + reason; bloki L=7, D=5, R=5"),
        ("target po joinie", "target_status, target_available, target_candidate_v2_pit_b"),
        ("diagnostyczny size", "year-specific quartile dodatnich current_t_assets; osobny missing bucket"),
    ],
    columns=["grupa", "pola / znaczenie"],
).set_index("grupa")
show("Artefakty", structure)
show("Schemat ramki analitycznej", schema)



Artefakty
                                                                                  jednostka pełny_kształt                                               rola_w_analizie
artefakt                                                                                                                                                               
historical universe v1.1.0             registrant–fiscal year; wszystkie statusy membership   103099 × 81                       upstream invariant; eligible membership
target PIT-B v1.0.0                                  company-year starego development frame   26917 × 802                          upstream invariant definicji targetu
raw X_t v1.0.0                                                        eligible company-year  64901 × 1072  metadata, 17 raw features, statusy, reason codes, provenance
audited universe × target application                                 eligible company-year   64901 × 832       target_status i target_candidate_v2_p

## 2. Populacja development, target i statusy X_t

Najpierw pokazana jest pełna eligible populacja development. Dopiero
później status `target_status == available` tworzy supervised ledger.
Niedostępnego targetu nie mapuje się na klasę 0.


In [3]:
split_summary = analysis_frame.groupby("split", sort=False).agg(
    eligible_n=("research_universe_company_year_id", "size"),
    target_available_n=("supervised_target_available", "sum"),
    target_coverage=("supervised_target_available", "mean"),
)
split_summary["target_coverage_pct"] = (100 * split_summary.pop("target_coverage")).round(2)

target_status_counts = pd.crosstab(
    analysis_frame["split"], analysis_frame["target_status"], margins=True
)
target_balance = supervised_frame.groupby("split", sort=False).agg(
    target_available_n=("target_candidate_v2_pit_b", "size"),
    negative_n=("target_candidate_v2_pit_b", lambda x: x.eq(0).sum()),
    positive_n=("target_candidate_v2_pit_b", lambda x: x.eq(1).sum()),
    positive_rate=("target_candidate_v2_pit_b", "mean"),
)
target_balance.loc["all"] = [
    len(supervised_frame),
    supervised_frame["target_candidate_v2_pit_b"].eq(0).sum(),
    supervised_frame["target_candidate_v2_pit_b"].eq(1).sum(),
    supervised_frame["target_candidate_v2_pit_b"].mean(),
]
target_balance["positive_rate_pct"] = (100 * target_balance.pop("positive_rate")).round(2)

x_status_counts = pd.crosstab(
    analysis_frame["split"], analysis_frame["x_t_status"]
).reindex(columns=STATUS_ORDER, fill_value=0)
x_status_share = (100 * x_status_counts.div(x_status_counts.sum(axis=1), axis=0)).round(2)

show("Liczebności train/validation i dostępność targetu", split_summary)
show("Target status — liczebności", target_status_counts)
show("Target balance wśród target_status == available", target_balance)
show("X_t status — liczebności", x_status_counts)
show("X_t status — procent w obrębie splitu", x_status_share)



Liczebności train/validation i dostępność targetu
            eligible_n  target_available_n  target_coverage_pct
split                                                          
train            47938               19784                41.27
validation        8965                3548                39.58

Target status — liczebności
target_status  ambiguous  available  hard_exclude  missing  not_computable    All
split                                                                            
train              11001      19784           208    12214            4731  47938
validation          2881       3548            28     2031             477   8965
All                13882      23332           236    14245            5208  56903

Target balance wśród target_status == available
            target_available_n  negative_n  positive_n  positive_rate_pct
split                                                                    
train                  19784.0     16137.0      3647.0   

## 3. Rozstrzygnięcie supervised ledger i charakterystyka grup X_t

Warunki bazowe są egzekwowane w tej kolejności:

1. `membership_status == eligible` — już zagwarantowane przez frozen raw `X_t` i assertion;
2. feature year 2011–2022;
3. `target_status == available` oraz etykieta dokładnie 0/1;
4. status X_t nie wpływa na sam supervised ledger; wpływa dopiero na
   możliwość zbudowania macierzy modelowej.

`available_core` oznacza komplet siedmiu cech poziomów L, ale nie
gwarantuje kompletu D i R. `partially_available` oznacza co najmniej
jedną dostępną frozen feature. Trzy pozostałe statusy mają zero
dostępnych frozen features.


In [4]:
def positive_assets_median_musd(group: pd.DataFrame) -> float:
    values = group.loc[group["current_t_assets_value"].gt(0), "current_t_assets_value"]
    return values.median() / 1_000_000 if len(values) else np.nan

profile_rows = []
for status in STATUS_ORDER:
    group = analysis_frame.loc[analysis_frame["x_t_status"].eq(status)]
    supervised = group.loc[group["supervised_target_available"]]
    profile_rows.append(
        {
            "x_t_status": status,
            "development_n": len(group),
            "train_n": group["split"].eq("train").sum(),
            "validation_n": group["split"].eq("validation").sum(),
            "target_available_n": len(supervised),
            "target_coverage_pct": round(100 * len(supervised) / len(group), 2),
            "positive_rate_pct": round(100 * supervised["target_candidate_v2_pit_b"].mean(), 2)
            if len(supervised) else np.nan,
            "positive_assets_median_MUSD": round(positive_assets_median_musd(group), 3),
            "mean_L_available": round(group["L_available_count"].mean(), 2),
            "mean_D_available": round(group["D_available_count"].mean(), 2),
            "mean_R_available": round(group["R_available_count"].mean(), 2),
            "mean_features_available": round(group["feature_available_count"].mean(), 2),
        }
    )
status_profile = pd.DataFrame(profile_rows).set_index("x_t_status")

supervised_status_counts = pd.crosstab(
    supervised_frame["split"], supervised_frame["x_t_status"], margins=True
).reindex(columns=STATUS_ORDER, fill_value=0)
supervised_class_by_status = pd.crosstab(
    supervised_frame["x_t_status"],
    supervised_frame["target_candidate_v2_pit_b"],
    margins=True,
).rename(columns={0.0: "target_0", 1.0: "target_1"})

show("Charakterystyka każdej grupy X_t — pełne development", status_profile)
show("Supervised ledger: status X_t według splitu", supervised_status_counts)
show("Supervised ledger: target balance według statusu X_t", supervised_class_by_status)



Charakterystyka każdej grupy X_t — pełne development
                        development_n  train_n  validation_n  target_available_n  target_coverage_pct  positive_rate_pct  positive_assets_median_MUSD  mean_L_available  mean_D_available  mean_R_available  mean_features_available
x_t_status                                                                                                                                                                                                                          
available_core                  45797    37622          8175               22679                49.52              19.10                      231.457              7.00              4.54              2.91                    14.45
partially_available              6086     5553           533                 539                 8.86              23.01                       28.587              4.64              2.87              1.47                     8.97
missing                       

### Przyczyny niedostępności

Pierwsza tabela pokazuje powód statusu na poziomie wiersza. Druga
pokazuje sześć najczęstszych kombinacji feature/status/reason w każdej
grupie na pełnym development — liczenie odbywa się przed filtrem
targetu, aby nie charakteryzować mechanizmu X wyłącznie na target
complete cases. Dla `available_core` są to luki w D/R, nie w L.


In [5]:
row_reasons = (
    analysis_frame.groupby(["x_t_status", "x_t_status_reason"], dropna=False)
    .size()
    .rename("observations")
    .reset_index()
)
row_reasons["share_within_status_pct"] = (
    100 * row_reasons["observations"]
    / row_reasons.groupby("x_t_status")["observations"].transform("sum")
).round(2)

feature_reason_records = []
for status in STATUS_ORDER:
    group = analysis_frame.loc[analysis_frame["x_t_status"].eq(status)]
    for feature in FEATURES:
        status_column = f"{feature}_status"
        reason_column = f"{feature}_reason"
        unavailable = group.loc[
            ~group[status_column].eq("available"), [status_column, reason_column]
        ]
        counts = unavailable.groupby([status_column, reason_column], dropna=False).size()
        for (feature_status, reason), observations in counts.items():
            feature_reason_records.append(
                {
                    "x_t_status": status,
                    "feature": feature,
                    "feature_status": feature_status,
                    "reason": reason,
                    "observations": observations,
                }
            )
feature_reasons = pd.DataFrame(feature_reason_records).sort_values(
    ["x_t_status", "observations", "feature"], ascending=[True, False, True]
)
feature_reasons["rank_within_x_status"] = (
    feature_reasons.groupby("x_t_status").cumcount() + 1
)
top_feature_reasons = feature_reasons.loc[
    feature_reasons["rank_within_x_status"].le(6)
].set_index(["x_t_status", "rank_within_x_status"])

show("Row-level X_t status reasons", row_reasons.set_index(["x_t_status", "x_t_status_reason"]))
show("Top feature-level unavailability reasons per X_t group", top_feature_reasons)



Row-level X_t status reasons
                                                                                   observations  share_within_status_pct
x_t_status             x_t_status_reason                                                                                
ambiguous              no_feature_available_and_ambiguity_present                           366                   100.00
available_core         all_core_level_features_available                                  45797                   100.00
missing                no_feature_available                                                 727                   100.00
not_available_non_xbrl frozen_universe_anchor_has_no_xbrl_submission                       3697                    94.14
                       separate_statement_scope_not_tagged_in_joint_xbrl_instance           230                     5.86
partially_available    at_least_one_feature_available                                      6086                   100.00

T

## 4. Czy dostępność X_t zależy od roku, sektora, wielkości i klasy targetu?

Raportowane są dwie granice:

- **core rate** — wszystkie siedem cech L (`available_core`);
- **any-feature rate** — co najmniej jedna frozen feature
  (`available_core` lub `partially_available`).

Kwartyle assets są wyłącznie diagnostyczne i liczone osobno w każdym
roku z dodatnich `current_t_assets_value`. Bucket
`missing_or_nonpositive` pozostaje jawny. Zależność any-feature od tego
bucketu jest częściowo mechaniczna, ponieważ log-assets samo jest cechą;
dlatego interpretacja wielkości opiera się głównie na core rate
warunkowym względem zaobserwowanych dodatnich assets.


In [6]:
def availability_summary(frame: pd.DataFrame, by: str) -> pd.DataFrame:
    result = frame.groupby(by, dropna=False, sort=True).agg(
        observations=("research_universe_company_year_id", "size"),
        core_n=("x_core_complete", "sum"),
        any_feature_n=("x_has_any_observed_feature", "sum"),
        partial_n=("x_t_status", lambda x: x.eq("partially_available").sum()),
        target_available_n=("supervised_target_available", "sum"),
    )
    result["core_rate_pct"] = (100 * result["core_n"] / result["observations"]).round(2)
    result["any_feature_rate_pct"] = (
        100 * result["any_feature_n"] / result["observations"]
    ).round(2)
    result["partial_rate_pct"] = (100 * result["partial_n"] / result["observations"]).round(2)
    result["target_coverage_pct"] = (
        100 * result["target_available_n"] / result["observations"]
    ).round(2)
    return result

by_year = availability_summary(analysis_frame, "feature_year")
by_sector = availability_summary(analysis_frame, "research_sector")
by_size = availability_summary(analysis_frame, "size_group").reindex(
    ["Q1", "Q2", "Q3", "Q4", "missing_or_nonpositive"]
)

by_class = supervised_frame.groupby("target_candidate_v2_pit_b").agg(
    observations=("research_universe_company_year_id", "size"),
    core_n=("x_core_complete", "sum"),
    any_feature_n=("x_has_any_observed_feature", "sum"),
    partial_n=("x_t_status", lambda x: x.eq("partially_available").sum()),
)
by_class["core_rate_pct"] = (100 * by_class["core_n"] / by_class["observations"]).round(2)
by_class["any_feature_rate_pct"] = (
    100 * by_class["any_feature_n"] / by_class["observations"]
).round(2)
by_class["partial_rate_pct"] = (100 * by_class["partial_n"] / by_class["observations"]).round(2)
by_class.index = ["target_0", "target_1"]

def cramers_v(left: pd.Series, right: pd.Series) -> float:
    observed = pd.crosstab(left, right).to_numpy(dtype=float)
    n = observed.sum()
    expected = observed.sum(axis=1)[:, None] * observed.sum(axis=0)[None, :] / n
    chi2 = np.divide(
        (observed - expected) ** 2,
        expected,
        out=np.zeros_like(observed),
        where=expected > 0,
    ).sum()
    denominator = n * min(observed.shape[0] - 1, observed.shape[1] - 1)
    return float(np.sqrt(chi2 / denominator)) if denominator > 0 else np.nan

dependence = pd.DataFrame(
    [
        ("feature_year", cramers_v(analysis_frame["feature_year"], analysis_frame["x_observation_profile"]), "development"),
        ("research_sector", cramers_v(analysis_frame["research_sector"], analysis_frame["x_observation_profile"]), "development"),
        ("size_group", cramers_v(analysis_frame["size_group"], analysis_frame["x_observation_profile"]), "development; partly mechanical"),
        ("target_class", cramers_v(supervised_frame["target_candidate_v2_pit_b"], supervised_frame["x_observation_profile"]), "target available only"),
    ],
    columns=["factor", "Cramers_V", "population"],
).set_index("factor")
dependence["Cramers_V"] = dependence["Cramers_V"].round(3)

show("Dostępność X_t według roku", by_year)
show("Dostępność X_t według sektora", by_sector)
show("Dostępność X_t według diagnostycznej wielkości", by_size)
show("Dostępność X_t według klasy targetu", by_class)
show("Siła związku z profilem X_t (Cramér's V; diagnostycznie)", dependence)



Dostępność X_t według roku
              observations  core_n  any_feature_n  partial_n  target_available_n  core_rate_pct  any_feature_rate_pct  partial_rate_pct  target_coverage_pct
feature_year                                                                                                                                                
2011                  5662    3406           3979        573                1926          60.16                 70.28             10.12                34.02
2012                  5364    3730           4806       1076                2317          69.54                 89.60             20.06                43.20
2013                  5262    4077           4817        740                2397          77.48                 91.54             14.06                45.55
2014                  5148    4076           4747        671                2301          79.18                 92.21             13.03                44.70
2015                  4851    

## 5. Pełny core kontra częściowy X_t i complete-case selection bias

W tym notebooku „pełny” oznacza status `available_core` (komplet L), a
nie komplet wszystkich 17 cech. Porównanie jest prowadzone w supervised
ledger, aby dotyczyło kandydatów do uczenia. Strict complete-case dla
L+D+R jest pokazany wyłącznie jako kontrfaktyczna polityka selekcji —
nie jako rekomendacja.


In [7]:
core_partial = supervised_frame.loc[
    supervised_frame["x_t_status"].isin(["available_core", "partially_available"])
]
comparison = core_partial.groupby("x_t_status", sort=False).agg(
    observations=("research_universe_company_year_id", "size"),
    train_n=("split", lambda x: x.eq("train").sum()),
    validation_n=("split", lambda x: x.eq("validation").sum()),
    validation_share_pct=("split", lambda x: round(100 * x.eq("validation").mean(), 2)),
    median_feature_year=("feature_year", "median"),
    positive_rate_pct=("target_candidate_v2_pit_b", lambda x: round(100 * x.mean(), 2)),
    log_assets_observed_n=("log_assets_t_value", "count"),
    log_assets_median=("log_assets_t_value", "median"),
    mean_L_available=("L_available_count", "mean"),
    mean_D_available=("D_available_count", "mean"),
    mean_R_available=("R_available_count", "mean"),
).round(3)

core_log_assets = core_partial.loc[
    core_partial["x_t_status"].eq("available_core"), "log_assets_t_value"
].dropna()
partial_log_assets = core_partial.loc[
    core_partial["x_t_status"].eq("partially_available"), "log_assets_t_value"
].dropna()
pooled_sd = np.sqrt((core_log_assets.var(ddof=1) + partial_log_assets.var(ddof=1)) / 2)
log_assets_smd_partial_vs_core = (
    partial_log_assets.mean() - core_log_assets.mean()
) / pooled_sd

feature_availability = []
for feature in FEATURES:
    row = {"feature": feature}
    for status in ["available_core", "partially_available"]:
        group = core_partial.loc[core_partial["x_t_status"].eq(status)]
        row[f"{status}_available_pct"] = round(
            100 * group[f"{feature}_status"].eq("available").mean(), 2
        )
    feature_availability.append(row)
feature_availability = pd.DataFrame(feature_availability).set_index("feature")

def policy_row(name: str, mask: pd.Series) -> dict:
    selected = supervised_frame.loc[mask]
    return {
        "policy": name,
        "n": len(selected),
        "retention_pct": round(100 * len(selected) / len(supervised_frame), 2),
        "train_n": selected["split"].eq("train").sum(),
        "validation_n": selected["split"].eq("validation").sum(),
        "positive_n": selected["target_candidate_v2_pit_b"].eq(1).sum(),
        "positive_rate_pct": round(100 * selected["target_candidate_v2_pit_b"].mean(), 2),
    }

scenarios = pd.DataFrame(
    [
        policy_row("all target-available; niezależnie od X", pd.Series(True, index=supervised_frame.index)),
        policy_row("recommended: available_core + partially_available", supervised_frame["x_has_any_observed_feature"]),
        policy_row("core-only: komplet L", supervised_frame["x_core_complete"]),
        policy_row(
            "strict L+D complete case",
            supervised_frame["L_available_count"].eq(7) & supervised_frame["D_available_count"].eq(5),
        ),
        policy_row(
            "strict L+D+R complete case",
            supervised_frame["L_available_count"].eq(7)
            & supervised_frame["D_available_count"].eq(5)
            & supervised_frame["R_available_count"].eq(5),
        ),
    ]
).set_index("policy")

sector_mix = (100 * pd.crosstab(
    core_partial["x_t_status"], core_partial["research_sector"], normalize="index"
)).round(2)

show("available_core vs partially_available — supervised ledger", comparison)
print(f"\nSMD log(assets), partially_available minus available_core: {log_assets_smd_partial_vs_core:.3f}")
show("Sektor — udział procentowy wewnątrz statusu", sector_mix)
show("Dostępność poszczególnych cech", feature_availability)
show("Kontrafaktyczne polityki kompletności", scenarios)



available_core vs partially_available — supervised ledger
                     observations  train_n  validation_n  validation_share_pct  median_feature_year  positive_rate_pct  log_assets_observed_n  log_assets_median  mean_L_available  mean_D_available  mean_R_available
x_t_status                                                                                                                                                                                                            
available_core              22679    19159          3520                 15.52               2016.0              19.10                  22679             19.864             7.000             4.471             4.577
partially_available           539      512            27                  5.01               2012.0              23.01                    533             18.767             4.503             1.920             1.918

SMD log(assets), partially_available minus available_core: -0.436

Sektor — udzi

### Ocena ryzyka complete-case selection bias

**Werdykt: wysokie ryzyko dla polityki complete-case.** Dostępność nie
jest losowa:

- any-feature rate rośnie z **70,28% w 2011** do **97,38% w 2022**, a
  core rate z **60,16%** do **91,88%**;
- core rate różni się sektorowo: od **73,46%** w Extended Candidate do
  **85,37%** w Industrials/Manufacturing;
- wśród obserwacji z dodatnimi assets core rate w Q1 wynosi **81,03%**,
  wobec około **91%** w Q2–Q4; bucket bez dodatnich zaobserwowanych
  assets praktycznie nie ma dostępnych cech;
- w supervised ledger `partially_available` jest wcześniejsze
  (mediana roku 2012 vs 2016), mniejsze (SMD log-assets = **−0,436**) i
  ma wyższy udział klasy dodatniej (**23,01% vs 19,10%**);
- zależność samego any-feature statusu od klasy targetu jest mała
  (99,52% dla klasy 0 i 99,46% dla klasy 1), ale nie niweluje zależności
  od czasu, sektora i wielkości;
- strict L+D+R complete case zachowałby tylko **79,39%** target-available
  ledger i wprowadziłby dodatkową selekcję zależną głównie od revenue.

Dodatkowo dostępność targetu wynosi tylko **41,00%** całej eligible
populacji development. Włączenie częściowego X ogranicza dodatkową
selekcję po stronie cech, lecz nie usuwa upstream target-selection i
informative-censoring risk.


## 6. Proponowana supervised sample policy

### 6.1. Główna polityka

Utworzyć dwa jawnie rozdzielone poziomy:

1. **Supervised ledger** — wszystkie `56 903` eligible development rows
   wraz z niezależnymi statusami targetu i X. Po warunku
   `target_status == available` ledger kandydatów do uczenia ma
   **23 332** wiersze (`19 784` train, `3 548` validation). Żadnego
   niedostępnego targetu nie mapować na 0.
2. **Primary model-eligible sample** — z target-available ledger dopuścić
   `available_core` i `partially_available`, bez progu liczby dostępnych
   cech i bez complete-case filter. Daje to **23 218** obserwacji:
   `19 671` train i `3 547` validation, z `4 455` klasami dodatnimi
   (19,19%). Braki poszczególnych cech pozostają NA i będą obsłużone
   dopiero przez osobno zatwierdzony, train-only preprocessing.

Raw `X_t` nie jest przycinany ani nadpisywany. Każdy modelowy artefakt
musi zachować `research_universe_company_year_id`, split i jawny
`supervised_sample_status/reason`.

### 6.2. Statusy dopuszczone

| `x_t_status` | Decyzja | Uzasadnienie |
|---|---|---|
| `available_core` | główna próba | pełny blok L; luki D/R nie mogą usuwać wiersza |
| `partially_available` | główna próba | co najmniej jedna wiarygodna frozen feature; pozwala uniknąć core complete-case selection |

### 6.3. Statusy poza główną macierzą modelową

| `x_t_status` | Target-available N | Decyzja | Uzasadnienie |
|---|---:|---|---|
| `missing` | 4 | wykluczyć z primary model matrix, zachować w ledger/audycie | zero obserwowanych frozen features |
| `ambiguous` | 7 | wykluczyć z primary model matrix, zachować w ledger/audycie | zero wiarygodnych features; semantyki nie wolno rozstrzygać przez imputację |
| `not_available_non_xbrl` | 103 | wykluczyć z primary model matrix, zachować w ledger/audycie | frozen X_t v1 nie ekstrahuje HTML/PDF; zero frozen features |

Wykluczenie tych 114 wierszy nie jest complete-case filterem: każdy z
nich ma **zero** dostępnych frozen predictors. Muszą jednak pozostać w
denominatorach coverage, opisie estimandu i sensitivity analysis.

### 6.4. Decyzje wymagające zatwierdzenia

1. **Granica primary sample:** zatwierdzić wspólną próbę
   `available_core + partially_available` (`N=23 218`) bez minimalnego
   progu liczby cech.
2. **Zero-feature rows:** zatwierdzić ich wyłączenie wyłącznie z primary
   matrix oraz obowiązkowe zachowanie w supervised ledger. Osobno
   zdecydować, czy później raportować missingness-only/intercept
   sensitivity, bounds/weighting albo zbudować przyszłe non-XBRL
   extension — bez zmiany frozen X_t v1.
3. **Wspólna próba dla L, L+D, L+D+R:** rekomendacja to wspólny sample i
   późniejszy train-only preprocessing, aby porównanie bloków nie
   mieszało wartości informacyjnej z inną populacją. Alternatywne
   block-complete-case samples mogą być wyłącznie sensitivity checks.
4. **Korekta selection bias:** przed CV zatwierdzić, czy poza pełnym
   raportem statusów będą stosowane wagi/bounds. Nie estymować ich ani
   nie wybierać na validation/test w tym notebooku.
5. **Missing indicators i ablation:** decyzja należy do osobnego projektu
   preprocessingu. Ten notebook ich nie tworzy.

**Na tym etapie nie wykonano preprocessingu, CV ani treningu modeli.**
